# Lucid Classifier Inspection

This notebook inspects the Lucid-style classifier used to predict sharing-score labels from solo-profile features.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

FEATURES_LUCID_FAITHFUL = [
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_drama",
    "amp_enabled",
]

FEATURES_EXTENDED = [
    *FEATURES_LUCID_FAITHFUL,
    "avg_smact",
    "avg_smocc",
    "horus_gpu_util_p95",
    "horus_gpu_util_max",
]

feature_table_path = Path("../results/lucid_feature_table.csv")
df = pd.read_csv(feature_table_path)
df.head()


,spec_name,spec_path,spec_key,peak_memory_mib,memory_fraction,horus_gpu_util_mean,horus_gpu_util_p95,horus_gpu_util_max,avg_smact,avg_smocc,...,representative_spec_path,lucid_mean_normalized_speed,lucid_std_normalized_speed,lucid_min_normalized_speed,lucid_max_normalized_speed,lucid_num_pair_observations,lucid_class,lucid_ss,lucid_label_source,lucid_label_usable
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_cifar100_bs128_20e_1gpu.yaml,860,0.020996,37.251852,44.0,56.0,0.226215,0.143911,...,evaluation/workloads/training/specs/yaml_thres...,0.950506,0.116516,0.574695,1.037478,14,tiny,0,measured_pairwise,True
1,efficientnet_imagenet_bs128_maxbatches1200.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_imagenet_bs128_1gpu.yaml,12732,0.310840,75.634328,85.0,86.0,0.605575,0.374403,...,evaluation/workloads/training/specs/yaml_thres...,0.833739,0.156536,0.490306,0.981883,13,jumbo,2,measured_pairwise,True
2,efficientnet_imagenet_bs64_maxbatches1200.yaml,evaluation/workloads/training/specs/yaml_thres...,efficientnet_imagenet_bs64_1gpu.yaml,6724,0.164160,77.585185,82.0,83.0,0.602948,0.363704,...,evaluation/workloads/training/specs/yaml_thres...,0.807846,0.157202,0.403917,0.983718,21,jumbo,2,measured_pairwise,True
3,llama3_width_8layer_wiki_bs1_1gpu_maxsteps2000...,evaluation/workloads/training/specs/yaml_thres...,llama3_width_8layer_wiki_bs1_1gpu.yaml,28712,0.700977,97.698795,99.0,99.0,0.295530,0.166651,...,evaluation/workloads/training/specs/yaml_thres...,0.895632,0.104657,0.656576,0.979835,9,medium,1,measured_pairwise,True
4,mobilenet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml_thres...,mobilenet_cifar100_bs128_20e_1gpu.yaml,634,0.015479,32.615385,40.0,46.0,0.157754,0.090031,...,evaluation/workloads/training/specs/yaml_thres...,0.898231,0.161183,0.486047,1.061015,15,medium,1,measured_pairwise,True


## Label coverage

In [17]:
print("Rows:", len(df))
print("\nUsable label counts:")
print(df["lucid_label_usable"].value_counts(dropna=False))

print("\nClass counts:")
print(df["lucid_class"].value_counts(dropna=False))

df[[
    "spec_key",
    "lucid_num_pair_observations",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
]].sort_values(["lucid_label_usable", "lucid_num_pair_observations"], ascending=[True, True])


Rows: 20

Usable label counts:
lucid_label_usable
True    20
Name: count, dtype: int64

Class counts:
lucid_class
jumbo     10
medium     7
tiny       3
Name: count, dtype: int64


,spec_key,lucid_num_pair_observations,lucid_label_usable,lucid_class,lucid_ss
5,mobilenet_cifar100_bs64_20e_1gpu.yaml,4,True,tiny,0
10,resnet34_cifar100_bs64_20e_1gpu.yaml,4,True,tiny,0
3,llama3_width_8layer_wiki_bs1_1gpu.yaml,9,True,medium,1
17,xception_imagenet_bs128_1gpu.yaml,10,True,medium,1
8,resnet18_cifar100_bs64_20e_1gpu.yaml,11,True,medium,1
11,resnet50_imagenet_bs128_1gpu.yaml,12,True,jumbo,2
1,efficientnet_imagenet_bs128_1gpu.yaml,13,True,jumbo,2
13,unet_voc_1gpu_10e_1gpu.yaml,13,True,jumbo,2
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,14,True,tiny,0
6,mobilenet_imagenet_bs128_1gpu.yaml,14,True,medium,1


## Train/evaluate classifier

In [18]:
def train_and_report(feature_cols, seed=42):
    train = df[df["lucid_label_usable"] == True].copy()
    X = train[feature_cols]
    y = train["lucid_ss"].astype(int)

    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=seed,
                    class_weight="balanced",
                    min_samples_leaf=1,
                ),
            ),
        ]
    )

    model.fit(X, y)

    if len(train) >= 3:
        pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
        print("Leave-one-out report")
        print(classification_report(y, pred, zero_division=0))
        print("Confusion matrix [0=tiny, 1=medium, 2=jumbo]")
        print(confusion_matrix(y, pred, labels=[0, 1, 2]))

    importances = model.named_steps["clf"].feature_importances_
    imp = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp = imp.sort_values("importance", ascending=False)

    return model, imp

model_lucid, imp_lucid = train_and_report(FEATURES_LUCID_FAITHFUL)
imp_lucid


Leave-one-out report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.50      0.43      0.46         7
           2       0.73      0.80      0.76        10

    accuracy                           0.55        20
   macro avg       0.41      0.41      0.41        20
weighted avg       0.54      0.55      0.54        20

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[0 2 1]
 [2 3 2]
 [1 1 8]]


,feature,importance
1,memory_fraction,0.408228
0,peak_memory_mib,0.373531
2,horus_gpu_util_mean,0.218241
3,amp_enabled,0.000000


## Extended feature set

In [19]:
model_ext, imp_ext = train_and_report(FEATURES_EXTENDED)
imp_ext


Leave-one-out report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.60      0.43      0.50         7
           2       0.75      0.90      0.82        10

    accuracy                           0.60        20
   macro avg       0.45      0.44      0.44        20
weighted avg       0.58      0.60      0.58        20

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[0 2 1]
 [2 3 2]
 [1 0 9]]


,feature,importance
0,peak_memory_mib,0.198195
1,memory_fraction,0.196874
8,horus_gpu_util_max,0.118821
7,horus_gpu_util_p95,0.110413
6,avg_drama,0.110298
4,avg_smact,0.098423
2,horus_gpu_util_mean,0.089463
5,avg_smocc,0.077513
3,amp_enabled,0.000000


## Compare measured and predicted labels

In [20]:
def predict_table(model, feature_cols, source_name):
    out = df.copy()
    pred = model.predict(out[feature_cols])
    proba = model.predict_proba(out[feature_cols])
    classes = list(model.named_steps["clf"].classes_)

    ss_to_class = {0: "tiny", 1: "medium", 2: "jumbo"}

    out[f"pred_ss_{source_name}"] = pred
    out[f"pred_class_{source_name}"] = [ss_to_class[int(x)] for x in pred]

    for i, cls in enumerate(classes):
        out[f"pred_proba_ss{int(cls)}_{source_name}"] = proba[:, i]

    return out

pred_lucid = predict_table(model_lucid, FEATURES_LUCID_FAITHFUL, "lucid")
pred_lucid[[
    "spec_key",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
    "pred_class_lucid",
    "pred_ss_lucid",
    "pred_proba_ss0_lucid",
    "pred_proba_ss1_lucid",
    "pred_proba_ss2_lucid",
]].sort_values("spec_key")


,spec_key,lucid_label_usable,lucid_class,lucid_ss,pred_class_lucid,pred_ss_lucid,pred_proba_ss0_lucid,pred_proba_ss1_lucid,pred_proba_ss2_lucid
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,True,tiny,0,tiny,0,0.735,0.150,0.115
1,efficientnet_imagenet_bs128_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.225,0.775
2,efficientnet_imagenet_bs64_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.000,1.000
3,llama3_width_8layer_wiki_bs1_1gpu.yaml,True,medium,1,medium,1,0.000,0.950,0.050
4,mobilenet_cifar100_bs128_20e_1gpu.yaml,True,medium,1,medium,1,0.220,0.770,0.010
5,mobilenet_cifar100_bs64_20e_1gpu.yaml,True,tiny,0,tiny,0,0.645,0.345,0.010
6,mobilenet_imagenet_bs128_1gpu.yaml,True,medium,1,medium,1,0.000,0.805,0.195
7,mobilenet_imagenet_bs64_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.015,0.985
8,resnet18_cifar100_bs64_20e_1gpu.yaml,True,medium,1,medium,1,0.205,0.785,0.010
9,resnet34_cifar100_bs128_20e_1gpu.yaml,True,jumbo,2,jumbo,2,0.345,0.015,0.640


## Export notebook predictions

In [21]:
output = Path("../results/lucid_classifier_predictions_notebook.csv")
pred_lucid.to_csv(output, index=False)
print(output)


../results/lucid_classifier_predictions_notebook.csv


In [22]:
from pathlib import Path
import yaml
import pandas as pd

ALL_SPEC_DIR = Path("../../workloads/training/specs/yaml")
OUTPUT_ALL_SPECS = Path("../results/lucid_predictions_all_specs.csv")

def canonical_spec_key(path):
    name = Path(path).name
    stem = Path(name).stem

    suffixes = [
        "_maxbatches1200",
        "_maxbatches600",
        "_maxbatches",
        "_maxsteps2000",
    ]
    for suffix in suffixes:
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break

    if not stem.endswith("_1gpu"):
        stem = f"{stem}_1gpu"

    return f"{stem}.yaml"

def command_has_amp(command):
    return int("--amp" in str(command).split())

def build_feature_row(spec_path, gpu_capacity_mib=40960):
    data = yaml.safe_load(spec_path.read_text()) or {}
    profile = data.get("profile", {}) or {}
    job = data.get("job", {}) or {}
    command = str(job.get("command", ""))

    peak_memory_mib = profile.get("peak_memory_mib")

    return {
        "spec_name": spec_path.name,
        "spec_path": str(spec_path),
        "spec_key": canonical_spec_key(spec_path),
        "peak_memory_mib": peak_memory_mib,
        "memory_fraction": None if peak_memory_mib is None else float(peak_memory_mib) / gpu_capacity_mib,
        "horus_gpu_util_mean": profile.get("horus_gpu_util_mean"),
        "horus_gpu_util_p95": profile.get("horus_gpu_util_p95"),
        "horus_gpu_util_max": profile.get("horus_gpu_util_max"),
        "avg_smact": profile.get("avg_smact"),
        "avg_smocc": profile.get("avg_smocc"),
        "avg_drama": profile.get("avg_drama"),
        "amp_enabled": command_has_amp(command),
    }

all_specs = sorted(ALL_SPEC_DIR.glob("*.yaml"))
all_features = pd.DataFrame([build_feature_row(p) for p in all_specs])

pred_ss = model_lucid.predict(all_features[FEATURES_LUCID_FAITHFUL])
pred_proba = model_lucid.predict_proba(all_features[FEATURES_LUCID_FAITHFUL])
classes = list(model_lucid.named_steps["clf"].classes_)

ss_to_class = {0: "tiny", 1: "medium", 2: "jumbo"}

all_predictions = all_features.copy()
all_predictions["lucid_pred_ss"] = pred_ss
all_predictions["lucid_pred_class"] = [ss_to_class[int(x)] for x in pred_ss]
all_predictions["lucid_pred_source"] = "predicted_classifier_lucid_faithful"

for i, cls in enumerate(classes):
    all_predictions[f"lucid_pred_proba_ss{int(cls)}"] = pred_proba[:, i]

all_predictions.to_csv(OUTPUT_ALL_SPECS, index=False)

print("rows:", len(all_predictions))
print("wrote:", OUTPUT_ALL_SPECS)
print(all_predictions[[
    "spec_name",
    "lucid_pred_class",
    "lucid_pred_ss",
    "lucid_pred_proba_ss0",
    "lucid_pred_proba_ss1",
    "lucid_pred_proba_ss2",
]].to_string(index=False))

rows: 53
wrote: ../results/lucid_predictions_all_specs.csv
                                spec_name lucid_pred_class  lucid_pred_ss  lucid_pred_proba_ss0  lucid_pred_proba_ss1  lucid_pred_proba_ss2
            bert_base_wiki_bs32_1gpu.yaml           medium              1                 0.000                 0.905                 0.095
            bert_large_wiki_bs8_1gpu.yaml            jumbo              2                 0.000                 0.225                 0.775
            dlrm_criteo_bs32768_1gpu.yaml            jumbo              2                 0.375                 0.110                 0.515
efficientnet_cifar100_bs128_20e_1gpu.yaml             tiny              0                 0.735                 0.150                 0.115
efficientnet_cifar100_bs128_50e_1gpu.yaml             tiny              0                 0.775                 0.195                 0.030
 efficientnet_cifar100_bs32_20e_1gpu.yaml           medium              1                 0.385      

In [23]:
proba_cols = ["lucid_pred_proba_ss0", "lucid_pred_proba_ss1", "lucid_pred_proba_ss2"]
all_predictions["max_proba"] = all_predictions[proba_cols].max(axis=1)

all_predictions.sort_values("max_proba")[[
    "spec_name",
    "lucid_pred_class",
    "lucid_pred_ss",
    "max_proba",
    *proba_cols,
]]

,spec_name,lucid_pred_class,lucid_pred_ss,max_proba,lucid_pred_proba_ss0,lucid_pred_proba_ss1,lucid_pred_proba_ss2
30,resnet18_cifar100_bs128_50e_1gpu.yaml,tiny,0,0.510,0.510,0.480,0.010
29,resnet18_cifar100_bs128_20e_1gpu.yaml,tiny,0,0.510,0.510,0.480,0.010
2,dlrm_criteo_bs32768_1gpu.yaml,jumbo,2,0.515,0.375,0.110,0.515
5,efficientnet_cifar100_bs32_20e_1gpu.yaml,medium,1,0.515,0.385,0.515,0.100
31,resnet18_cifar100_bs32_20e_1gpu.yaml,medium,1,0.515,0.470,0.515,0.015
19,mnist_bs32_1gpu.yaml,jumbo,2,0.515,0.375,0.110,0.515
32,resnet18_cifar100_bs32_50e_1gpu.yaml,medium,1,0.515,0.470,0.515,0.015
16,inception_imagenet_bs64_1gpu.yaml,medium,1,0.575,0.000,0.575,0.425
8,efficientnet_cifar100_bs64_50e_1gpu.yaml,medium,1,0.580,0.320,0.580,0.100
7,efficientnet_cifar100_bs64_20e_1gpu.yaml,medium,1,0.580,0.320,0.580,0.100


## Tiny-class safeguards
The classifier has weak Tiny support, so we do not blindly trust classifier outputs for low-resource jobs.

In [24]:
train = df[df["lucid_label_usable"] == True].copy()

print("Training class counts:")
print(train["lucid_class"].value_counts().to_string())

tiny_train = train[train["lucid_ss"] == 0]
print("\nTiny training examples:")
display(tiny_train[[
    "spec_key",
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_mean_normalized_speed",
    "lucid_num_pair_observations",
]])


Training class counts:
lucid_class
jumbo     10
medium     7
tiny       3

Tiny training examples:


,spec_key,peak_memory_mib,memory_fraction,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_mean_normalized_speed,lucid_num_pair_observations
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,860,0.020996,37.251852,0.226215,0.143911,0.020215,0.950506,14
5,mobilenet_cifar100_bs64_20e_1gpu.yaml,602,0.014697,30.969231,0.118638,0.061992,0.003038,1.063400,4
10,resnet34_cifar100_bs64_20e_1gpu.yaml,1018,0.024854,36.308271,0.179992,0.072256,0.060128,1.023268,4


In [25]:
# Conservative tiny-like heuristic:
# A workload is "tiny-like" if it falls below the measured Tiny envelope
# for memory and GPU utilization.

tiny_memory_max = tiny_train["peak_memory_mib"].max()
tiny_util_max = tiny_train["horus_gpu_util_mean"].max()

print("Tiny envelope:")
print("max peak_memory_mib:", tiny_memory_max)
print("max horus_gpu_util_mean:", tiny_util_max)

all_predictions["tiny_like_by_profile"] = (
    (all_predictions["peak_memory_mib"] <= tiny_memory_max)
    & (all_predictions["horus_gpu_util_mean"] <= tiny_util_max)
)

tiny_conflicts = all_predictions[
    (all_predictions["tiny_like_by_profile"])
    & (all_predictions["lucid_pred_ss"] != 0)
].copy()

print("Tiny-like workloads predicted as non-tiny:", len(tiny_conflicts))

display(tiny_conflicts[[
    "spec_name",
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_pred_class",
    "lucid_pred_ss",
    "lucid_pred_proba_ss0",
    "lucid_pred_proba_ss1",
    "lucid_pred_proba_ss2",
]])

Tiny envelope:
max peak_memory_mib: 1018
max horus_gpu_util_mean: 37.25185185185185
Tiny-like workloads predicted as non-tiny: 10


,spec_name,peak_memory_mib,memory_fraction,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_pred_class,lucid_pred_ss,lucid_pred_proba_ss0,lucid_pred_proba_ss1,lucid_pred_proba_ss2
5,efficientnet_cifar100_bs32_20e_1gpu.yaml,668,0.016309,36.830769,0.159577,0.087654,0.008246,medium,1,0.385,0.515,0.100
6,efficientnet_cifar100_bs32_50e_1gpu.yaml,668,0.016309,34.931298,0.152466,0.084298,0.007962,medium,1,0.240,0.750,0.010
7,efficientnet_cifar100_bs64_20e_1gpu.yaml,724,0.017676,37.187970,0.187820,0.112038,0.011053,medium,1,0.320,0.580,0.100
8,efficientnet_cifar100_bs64_50e_1gpu.yaml,724,0.017676,36.443609,0.186271,0.110805,0.010835,medium,1,0.320,0.580,0.100
20,mobilenet_cifar100_bs128_20e_1gpu.yaml,634,0.015479,32.615385,0.157754,0.090031,0.005069,medium,1,0.220,0.770,0.010
21,mobilenet_cifar100_bs128_50e_1gpu.yaml,634,0.015479,31.992366,0.154870,0.086794,0.004511,medium,1,0.220,0.770,0.010
31,resnet18_cifar100_bs32_20e_1gpu.yaml,790,0.019287,36.122137,0.170344,0.062206,0.048733,medium,1,0.470,0.515,0.015
32,resnet18_cifar100_bs32_50e_1gpu.yaml,790,0.019287,35.863636,0.173242,0.063311,0.049705,medium,1,0.470,0.515,0.015
33,resnet18_cifar100_bs64_20e_1gpu.yaml,798,0.019482,34.338346,0.166519,0.062098,0.049241,medium,1,0.205,0.785,0.010
34,resnet18_cifar100_bs64_50e_1gpu.yaml,798,0.019482,34.601504,0.168286,0.063120,0.051835,medium,1,0.255,0.735,0.010


In [26]:
all_predictions["lucid_final_ss"] = all_predictions["lucid_pred_ss"]
all_predictions["lucid_final_class"] = all_predictions["lucid_pred_class"]
all_predictions["lucid_final_source"] = all_predictions["lucid_pred_source"]

# Apply conservative Tiny safeguard.
tiny_conflict_mask = (
    all_predictions["tiny_like_by_profile"]
    & (all_predictions["lucid_pred_ss"] != 0)
)

all_predictions.loc[tiny_conflict_mask, "lucid_final_ss"] = 0
all_predictions.loc[tiny_conflict_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[tiny_conflict_mask, "lucid_final_source"] = (
    "tiny_profile_safeguard_after_classifier"
)

print("Final class counts after safeguard:")
print(all_predictions["lucid_final_class"].value_counts().to_string())

display(all_predictions[[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "tiny_like_by_profile",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]].sort_values(["lucid_final_class", "peak_memory_mib"]))

Final class counts after safeguard:
lucid_final_class
tiny      22
jumbo     19
medium    12


,spec_name,peak_memory_mib,horus_gpu_util_mean,tiny_like_by_profile,lucid_pred_class,lucid_final_class,lucid_final_source
35,resnet34_cifar100_bs128_20e_1gpu.yaml,1076,36.476562,False,jumbo,jumbo,predicted_classifier_lucid_faithful
36,resnet34_cifar100_bs128_50e_1gpu.yaml,1076,39.088889,False,jumbo,jumbo,predicted_classifier_lucid_faithful
19,mnist_bs32_1gpu.yaml,1098,16.285714,False,jumbo,jumbo,predicted_classifier_lucid_faithful
2,dlrm_criteo_bs32768_1gpu.yaml,1364,10.477612,False,jumbo,jumbo,predicted_classifier_lucid_faithful
27,mobilenet_imagenet_bs32_1gpu.yaml,3354,63.614815,False,jumbo,jumbo,predicted_classifier_lucid_faithful
10,efficientnet_imagenet_bs32_1gpu.yaml,3760,62.696296,False,jumbo,jumbo,predicted_classifier_lucid_faithful
42,resnet50_imagenet_bs32_1gpu.yaml,3944,77.425373,False,jumbo,jumbo,predicted_classifier_lucid_faithful
15,inception_imagenet_bs32_1gpu.yaml,5246,75.451852,False,jumbo,jumbo,predicted_classifier_lucid_faithful
44,unet_voc_1gpu.yaml,5636,84.201493,False,jumbo,jumbo,predicted_classifier_lucid_faithful
49,xception_imagenet_bs32_1gpu.yaml,5922,82.395522,False,jumbo,jumbo,predicted_classifier_lucid_faithful


In [27]:
OUTPUT_FINAL_ALL_SPECS = Path("../results/lucid_predictions_all_specs_with_tiny_safeguard.csv")

all_predictions.to_csv(OUTPUT_FINAL_ALL_SPECS, index=False)

print("wrote:", OUTPUT_FINAL_ALL_SPECS)

wrote: ../results/lucid_predictions_all_specs_with_tiny_safeguard.csv


Because the measured Lucid labels contain relatively few Tiny examples, the classifier may be biased against predicting Tiny. To avoid overclassifying low-resource workloads as Medium/Jumbo, we add a conservative post-classification safeguard. If a workload falls inside the measured Tiny envelope for both peak GPU memory and mean GPU utilization, but the classifier predicts a higher sharing score, we mark it as Tiny and record the source as `tiny_profile_safeguard_after_classifier`. This safeguard is intentionally conservative and transparent.

In [28]:
# Additional conservative safeguard:
# If a classifier-only workload has very low memory footprint and very low
# mean GPU utilization, avoid assigning it Jumbo. We cap it at Medium.
low_resource_mask = (
    (all_predictions["peak_memory_mib"] <= 2048)
    & (all_predictions["horus_gpu_util_mean"] <= 20)
    & (all_predictions["lucid_final_ss"] == 2)
)

all_predictions.loc[low_resource_mask, "lucid_final_ss"] = 1
all_predictions.loc[low_resource_mask, "lucid_final_class"] = "medium"
all_predictions.loc[low_resource_mask, "lucid_final_source"] = (
    "low_resource_safeguard_after_classifier"
)

display(all_predictions[low_resource_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,lucid_pred_class,lucid_final_class,lucid_final_source
2,dlrm_criteo_bs32768_1gpu.yaml,1364,10.477612,jumbo,medium,low_resource_safeguard_after_classifier
19,mnist_bs32_1gpu.yaml,1098,16.285714,jumbo,medium,low_resource_safeguard_after_classifier


In [29]:
# Conservative low-resource safeguard:
# The classifier has weak Tiny support, so very-low-resource classifier-only
# workloads should not be promoted to Medium/Jumbo solely by the classifier.
low_resource_tiny_mask = (
    (all_predictions["peak_memory_mib"] <= 2048)
    & (all_predictions["horus_gpu_util_mean"] <= 20)
)

all_predictions.loc[low_resource_tiny_mask, "lucid_final_ss"] = 0
all_predictions.loc[low_resource_tiny_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[low_resource_tiny_mask, "lucid_final_source"] = (
    "low_resource_tiny_safeguard_after_classifier"
)

display(all_predictions[low_resource_tiny_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_pred_class,lucid_final_class,lucid_final_source
2,dlrm_criteo_bs32768_1gpu.yaml,1364,10.477612,0.139478,0.046537,0.027134,jumbo,tiny,low_resource_tiny_safeguard_after_classifier
19,mnist_bs32_1gpu.yaml,1098,16.285714,0.079077,0.029780,0.012725,jumbo,tiny,low_resource_tiny_safeguard_after_classifier


In [30]:
# Manual transparent override for known toy/sanity workloads.
manual_tiny_patterns = ["mnist"]

manual_tiny_mask = all_predictions["spec_name"].str.lower().apply(
    lambda name: any(pattern in name for pattern in manual_tiny_patterns)
)

all_predictions.loc[manual_tiny_mask, "lucid_final_ss"] = 0
all_predictions.loc[manual_tiny_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[manual_tiny_mask, "lucid_final_source"] = (
    "manual_tiny_workload_override"
)

display(all_predictions[manual_tiny_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,lucid_pred_class,lucid_final_class,lucid_final_source
19,mnist_bs32_1gpu.yaml,1098,16.285714,jumbo,tiny,manual_tiny_workload_override


Because the measured Lucid labels contain relatively few Tiny examples, the classifier tends to overestimate sharing scores for very small workloads. We therefore apply two transparent safeguards. First, classifier-only workloads with both low peak GPU memory and low mean GPU utilization are labeled Tiny. Second, known toy/sanity workloads such as MNIST are explicitly marked Tiny. These safeguards are recorded in the label source field and are not hidden as classifier outputs.

In [31]:
OUTPUT_FINAL_ALL_SPECS = Path("../results/lucid_predictions_all_specs_with_tiny_safeguard.csv")
all_predictions.to_csv(OUTPUT_FINAL_ALL_SPECS, index=False)
print("wrote:", OUTPUT_FINAL_ALL_SPECS)

wrote: ../results/lucid_predictions_all_specs_with_tiny_safeguard.csv
